# LSN-002｜从浮点数到有限位宽
**对应：RMD-002 · FR1/FR2 · T-005~T-006**

## 本课问题
Python 里的 `0.1`、`0.9` 看起来理所当然，但真实 FPGA 里的数字需要有限数量的 bit。我们怎样把连续数值映射成有限位宽？

> 本课唯一主要新概念：**fixed-point / quantization。**

## 直觉
把 fixed-point 想成一把只有有限刻度的尺。`frac_bits` 决定刻度有多细，`total_bits` 决定这把尺能量多大的范围。

我们先在 Python 里模拟这种限制，再让 RTL 复现同样规则。

In [ ]:
def quantize(x, total_bits=8, frac_bits=4):
    scale = 1 << frac_bits
    min_i = -(1 << (total_bits - 1))
    max_i = (1 << (total_bits - 1)) - 1
    q = round(x * scale)
    q = max(min_i, min(max_i, q))
    return q / scale

for x in [0.1, 0.22, 0.9, 1.7, -0.3]:
    print(x, '->', quantize(x, 8, 4))

In [ ]:
def run_lif_quantized(inputs, total_bits, frac_bits, alpha=0.9, threshold=1.0, reset=0.0):
    q = lambda x: quantize(x, total_bits, frac_bits)
    v = q(0.0)
    spikes = []
    trace = []
    for t, current in enumerate(inputs):
        v = q(q(alpha) * v + q(current))
        spike = v >= q(threshold)
        if spike:
            spikes.append(t)
            v = q(reset)
        trace.append(v)
    return spikes, trace

inputs = [0.22] * 30
for fmt in [(8, 4), (12, 8), (16, 12)]:
    spikes, trace = run_lif_quantized(inputs, *fmt)
    print(f'Q format total={fmt[0]}, frac={fmt[1]} -> spikes {spikes}')

## Observe
- 位宽变小时，哪些参数被改得最多？
- spike timing 会不会变化？
- saturation 和 wraparound 为什么会造成完全不同的错误？

这里不急着选最终 Q-format；目标是先看见**数值表示本身也是设计参数**。

## AI Task
让 AI 帮你增加参数扫描、误差表或 spike-time comparison。要求它明确写出 rounding 与 saturation 规则。

## Human Check
- integer bits 与 fractional bits 分别解决什么问题？
- 为什么 accumulator 可能需要比 membrane/weight 更宽？
- 两个 fixed-point 实现数值接近，是否一定 spike sequence 相同？

## Engineering Handoff
成熟实现进入 `python/reference/lif_fixed.py`，数值选择写入 MDD/TDD。

## Exit Ticket
你能解释 scale、rounding、saturation，并用实验说明位宽如何影响神经元行为。